# 20 AG extract: the ReEDS outputs that the paper-AG ledger reads

**Purpose.** Paper AG builds one ledger of money flows and emissions changes, each measured as a
policy run minus its baseline (`docs/protocol.md`, signed 2026-10-28). This notebook is the only
AG notebook that opens a ReEDS output file. It writes two kinds of extract to
`results/extracts_ag/`, and it never touches paper 1's extracts in `results/extracts/`:

1. **Linear extracts.** `data/extract_spec_ag.csv` names every report key that the ledger reads and
   that can be filtered or summed directly: revenue, system cost and credits by technology,
   capacity, generation, emissions, gas price and quantity, hurdle charges and the value of new
   builds. Unlike paper 1's spec, these rows keep every technology, not only nuclear.
   `nlpaper.h5io.extract(spec, out=...)` writes them, with their own `manifest.csv`.
2. **Derived extracts.** Payments and congestion rent are sums of price × quantity over regions and
   timeslices. They must be formed before any aggregation, so the spec cannot express them.
   `aglib.reduce.reduce_run` forms them per run, and this notebook writes them to
   `<batch>/derived_ledger.parquet`. The item definitions are in the module docstring and protocol §4.
3. **Census-division extracts.** `aglib.reduce.reduce_run_cendiv` keeps the census division of each
   zone before it sums: consumer payments, owner revenue and costs by technology group, and system
   cost. They go to `<batch>/derived_cendiv.parquet` (protocol amendment A16).
4. **REC extracts.** `aglib.reduce.reduce_run_recs` splits the REC trade wedge by origin for the stage-1
   runs and the baselines (§6). They go to `<batch>/derived_recs.parquet`.

A derived run that is already in its file is not reduced again. To rebuild after a change to a
reducer, delete that parquet file first.

**Scope of this run.** The linear extracts cover every run of the eight batches. The derived
extracts cover the stage-1 runs (the six P50 SMR mandate runs) and the 19 no-mandate baselines
(protocol §2 and §7). Stage 2 (Phase 4 of the plan) adds the remaining runs with the same function.

**Rules.** This notebook reads the archive and never writes to it. It sets `NL_REEDS_OUTPUTS`
inside its own kernel only. Notebook 01 must never run with that variable set (kit README).


In [1]:
import sys
from pathlib import Path
_p = Path.cwd().resolve()
while not (_p / "nlpaper").is_dir():           # find z-ethan/ from wherever the kernel starts
    _p = _p.parent
sys.path.insert(0, str(_p))
import os
import time

import pandas as pd

import nlpaper
from nlpaper import paths, h5io
from aglib import reduce as agr
nlpaper.setup(threads=4)
pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 200)

nlpaper setup: python 3.12.3 | numpy 1.26.4 | pandas 2.2.2 | matplotlib 3.8.4 | threads 4
determinism: seeds are explicit numpy Generators in each cell; results do not depend on PYTHONHASHSEED.


## 1. The archive

The public archive layout (`<batch>/<prefix><case>_outputs.h5`) is what `h5io.run_file` expects.
If `NL_REEDS_OUTPUTS` is unset, this cell sets it, for this kernel only, to the public archive on
drive D.

In [2]:
PUBLIC_ARCHIVE = Path("D:/ReEDS files/nuclear-learning/public_archive/nuclear-learning-reeds-outputs")
os.environ.setdefault(paths.ENV_OUTPUTS, str(PUBLIC_ARCHIVE))
ARCHIVE = paths.reeds_outputs()
assert ARCHIVE is not None, "the ReEDS output archive is required for notebook 20"
print(f"reading ReEDS outputs from {ARCHIVE}")
EXTRACTS_AG = paths.RESULTS / "extracts_ag"
print(f"writing AG extracts to {paths.rel(EXTRACTS_AG)}")

reading ReEDS outputs from D:\ReEDS files\nuclear-learning\public_archive\nuclear-learning-reeds-outputs
writing AG extracts to z-ethan\results\extracts_ag


## 2. The linear extracts

Each spec row names a key, the batches, a row filter, an aggregation and the dimensions kept
(the column rules are those of `h5io._apply`). An extract is skipped when its parquet file exists
and its spec hash is unchanged.

In [3]:
SPEC = h5io.read_spec(paths.DATA / "extract_spec_ag.csv")
pairs = [(b, r["key"]) for _, r in SPEC.iterrows() for b in h5io._row_batches(r)]
assert len(pairs) == len(set(pairs)), "each (batch, key) appears in exactly one spec row"
print(f"{len(SPEC)} spec rows, {len(pairs)} (batch, key) extracts")
display(SPEC[["key", "batches", "tech_filter", "aggregate_to", "dims_kept"]])

38 spec rows, 304 (batch, key) extracts


,key,batches,tech_filter,aggregate_to,dims_kept
0,revenue_nat,all,,,
1,systemcost,all,,,
2,systemcost_techba,all,,national,
3,capex_ivrt,all,,national,i;t
4,cap_new_ivrt,all,,national,i;t
5,cap_nat,all,,,
6,cap_new_ann_nat,all,,,
7,ret_ann_nat,all,,,
8,gen_ann_nat,all,,,
9,nuclear_cap_price,all,,,


In [4]:
t0 = time.time()
MAN = h5io.extract(SPEC, out=EXTRACTS_AG, verbose=False)
print(f"linear extracts done in {(time.time() - t0) / 60:.1f} min")
summary = MAN.groupby("batch").agg(keys=("key", "nunique"), max_cases=("n_cases", "max"),
                                   MB=("bytes", lambda s: s.sum() / 1e6)).round(1)
display(summary)
missing = MAN[MAN["n_missing"] > 0][["batch", "key", "n_cases", "n_missing"]]
print("extracts with runs that lack the key (expected: the mandate shadow price in credit runs, "
      "co2_price outside the cap runs, the two unreturned batch-3 runs):")
display(missing)

WARNING smr100/co2_price: no run carries this key; nothing written
WARNING step4/co2_price: no run carries this key; nothing written
WARNING itcfb/co2_price: no run carries this key; nothing written
WARNING itcfbm/nuclear_cap_price: no run carries this key; nothing written
WARNING itcfbm/co2_price: no run carries this key; nothing written
WARNING ratedesign/co2_price: no run carries this key; nothing written
WARNING ratedesign2/co2_price: no run carries this key; nothing written


WARNING batch2/nuclear_cap_price: no run carries this key; nothing written
WARNING batch2/co2_price: no run carries this key; nothing written
linear extracts done in 0.0 min


,keys,max_cases,MB
batch,,,
ag,38,23,7.5
batch2,36,89,23.0
batch3,38,149,43.0
itcfb,37,19,5.1
itcfbm,36,24,6.9
ratedesign,37,66,18.5
ratedesign2,37,93,24.8
smr100,37,25,7.2
step4,37,120,34.9


extracts with runs that lack the key (expected: the mandate shadow price in credit runs, co2_price outside the cap runs, the two unreturned batch-3 runs):


,batch,key,n_cases,n_missing
0,ag,cap_nat,23,5
1,ag,cap_new_ann_nat,23,5
2,ag,cap_new_ivrt,23,5
3,ag,capex_ivrt,23,5
4,ag,co2_price,1,27
5,ag,cost_scale,23,5
6,ag,dropped_load,23,5
7,ag,emit_irt,23,5
8,ag,emit_nat,23,5
9,ag,emit_nat_tech,23,5


## 3. The derived extracts (stage 1 and the baselines)

The stage-1 runs are the six P50 SMR mandate runs. The 19 baselines are the no-mandate runs that
paper 1's V1-f selected because they build no SMR (`v1f_common.py:29-38` in paper 1): 11 in the
reference market and 8 under the emissions constraint.

In [5]:
STAGE1 = {
    "smr100": [f"smr100_{t}_p50" for t in ["eia", "aj", "iaea", "mck", "cop28", "eo"]],
    "itcfb": ["fbR_none", "fbR_itc30"],
    "batch2": [f"bx_{x}_f50" for x in ["aj_p75", "cop28_p50", "cop28_p75", "eia_p75", "eo_p75",
                                       "iaea_p50", "iaea_p75", "mck_p75"]],
    "ratedesign2": ["ax_eia_p95_c60"],
    "batch3": [f"bx_{x}_f50_co2cap" for x in ["aj_p75", "cop28_p50", "cop28_p75", "eia_p75",
                                              "eo_p75", "iaea_p50", "iaea_p75", "mck_p75"]],
}
assert sum(len(v) for v in STAGE1.values()) == 6 + 19
t0 = time.time()
for b, cs in STAGE1.items():
    dst = agr.reduce_cases(b, cs, out=EXTRACTS_AG, verbose=False, skip_existing=True)
    print(f"{b}: {len(cs)} runs -> {paths.rel(dst)}")
print(f"derived extracts done in {(time.time() - t0) / 60:.1f} min")

smr100: 6 runs -> z-ethan\results\extracts_ag\smr100\derived_ledger.parquet
itcfb: 2 runs -> z-ethan\results\extracts_ag\itcfb\derived_ledger.parquet
batch2: 8 runs -> z-ethan\results\extracts_ag\batch2\derived_ledger.parquet
ratedesign2: 1 runs -> z-ethan\results\extracts_ag\ratedesign2\derived_ledger.parquet
batch3: 8 runs -> z-ethan\results\extracts_ag\batch3\derived_ledger.parquet
derived extracts done in 0.0 min


In [6]:
rows = []
for b, cs in STAGE1.items():
    d = pd.read_parquet(EXTRACTS_AG / b / "derived_ledger.parquet")
    d = d[d["case"].isin(cs)]
    rows.append(d.groupby("case").agg(items=("item", "nunique"), first_t=("t", "min"),
                                      last_t=("t", "max")).assign(batch=b))
check = pd.concat(rows)
display(check)
assert set(check.index) == {c for cs in STAGE1.values() for c in cs}, "every stage-1 run reduced"
assert (check["last_t"] == 2050).all(), "every run reaches 2050"

,items,first_t,last_t,batch
case,,,,
smr100_aj_p50,13,2010,2050,smr100
smr100_cop28_p50,13,2010,2050,smr100
smr100_eia_p50,13,2010,2050,smr100
smr100_eo_p50,13,2010,2050,smr100
smr100_iaea_p50,13,2010,2050,smr100
smr100_mck_p50,13,2010,2050,smr100
fbR_itc30,13,2010,2050,itcfb
fbR_none,13,2010,2050,itcfb
bx_aj_p75_f50,13,2010,2050,batch2


## 4. The derived extracts for stage 2

The stage-2 run sets follow protocol §2. They are selected from the case names that the linear
extracts hold, so a run that never returned is simply absent.

| Stage | Runs |
|---|---|
| 2a | `smr100_{t}_{p05,p95}` (smr100), `smr100_{t}_{p25,p75}` (ratedesign), `ma_{t}_d{draw}` (ratedesign2) |
| 2b | `smr100_{t}_{p05,p50,p95}_{cond}` (step4), `smr100_{t}_{p25,p75}_{cond}` (batch3) |
| 2c | `smr100_{t}_{pNN}_co2cap` (batch3) |
| 2d | `fb{A,B,C,K}_*` (itcfb), `fbBm*`, `fbCm*` (itcfbm) |


In [7]:
import re
T6 = "(eia|aj|iaea|mck|cop28|eo)"
COND = "(gaslo|gashi|demhi|relo|rehi|translim)"
STAGE2_RULES = {
    "2a": [("smr100", rf"smr100_{T6}_p(05|95)"), ("ratedesign", rf"smr100_{T6}_p(25|75)"),
           ("ratedesign2", rf"ma_{T6}_d\d+")],
    "2b": [("step4", rf"smr100_{T6}_p(05|50|95)_{COND}"), ("batch3", rf"smr100_{T6}_p(25|75)_{COND}")],
    "2c": [("batch3", rf"smr100_{T6}_p\d\d_co2cap")],
    "2d": [("itcfb", r"fb[ABCK]_.+"), ("itcfbm", r"fb[BC]m\d\d_.+")],
}
cases_in = lambda b: sorted(pd.read_parquet(EXTRACTS_AG / b / "systemcost.parquet", columns=["case"])["case"].unique())
STAGE2 = {}
for st, rules in STAGE2_RULES.items():
    for b, rx in rules:
        cs = [c for c in cases_in(b) if re.fullmatch(rx, c)]
        STAGE2.setdefault(b, []).extend(cs)
        print(f"{st} {b:12s} {rx:45s} {len(cs)} runs")
n2 = sum(len(v) for v in STAGE2.values())
print("stage-2 runs:", n2)
assert n2 == 60 + 178 + 30 + 40, n2

2a smr100       smr100_(eia|aj|iaea|mck|cop28|eo)_p(05|95)    12 runs
2a ratedesign   smr100_(eia|aj|iaea|mck|cop28|eo)_p(25|75)    12 runs
2a ratedesign2  ma_(eia|aj|iaea|mck|cop28|eo)_d\d+            36 runs
2b step4        smr100_(eia|aj|iaea|mck|cop28|eo)_p(05|50|95)_(gaslo|gashi|demhi|relo|rehi|translim) 108 runs
2b batch3       smr100_(eia|aj|iaea|mck|cop28|eo)_p(25|75)_(gaslo|gashi|demhi|relo|rehi|translim) 70 runs
2c batch3       smr100_(eia|aj|iaea|mck|cop28|eo)_p\d\d_co2cap 30 runs
2d itcfb        fb[ABCK]_.+                                   16 runs
2d itcfbm       fb[BC]m\d\d_.+                                24 runs
stage-2 runs: 308


In [8]:
t0 = time.time()
for b, cs in STAGE2.items():
    t1 = time.time()
    dst = agr.reduce_cases(b, cs, out=EXTRACTS_AG, verbose=False, skip_existing=True)
    print(f"{b}: {len(cs)} runs -> {paths.rel(dst)} ({(time.time() - t1) / 60:.1f} min)", flush=True)
print(f"stage-2 derived extracts done in {(time.time() - t0) / 60:.1f} min")

smr100: 12 runs -> z-ethan\results\extracts_ag\smr100\derived_ledger.parquet (0.0 min)


ratedesign: 12 runs -> z-ethan\results\extracts_ag\ratedesign\derived_ledger.parquet (0.0 min)


ratedesign2: 36 runs -> z-ethan\results\extracts_ag\ratedesign2\derived_ledger.parquet (0.0 min)


step4: 108 runs -> z-ethan\results\extracts_ag\step4\derived_ledger.parquet (0.0 min)


batch3: 100 runs -> z-ethan\results\extracts_ag\batch3\derived_ledger.parquet (0.0 min)


itcfb: 16 runs -> z-ethan\results\extracts_ag\itcfb\derived_ledger.parquet (0.0 min)


itcfbm: 24 runs -> z-ethan\results\extracts_ag\itcfbm\derived_ledger.parquet (0.0 min)


stage-2 derived extracts done in 0.0 min


In [9]:
rows = []
for b, cs in STAGE2.items():
    d = pd.read_parquet(EXTRACTS_AG / b / "derived_ledger.parquet")
    d = d[d["case"].isin(cs)]
    rows.append(d.groupby("case").agg(items=("item", "nunique"), last_t=("t", "max")).assign(batch=b))
check2 = pd.concat(rows)
print(check2.groupby("batch").agg(runs=("items", "size"), items_min=("items", "min"), last_t_min=("last_t", "min")))
assert len(check2) == n2, "every stage-2 run reduced"
assert (check2["last_t"] == 2050).all(), "every run reaches 2050"

             runs  items_min  last_t_min
batch                                   
batch3        100         13        2050
itcfb          16         13        2050
itcfbm         24         13        2050
ratedesign     12         13        2050
ratedesign2    36         13        2050
smr100         12         13        2050
step4         108         13        2050


## 5. The census-division extracts

The census-division readouts (protocol amendment A16) need prices × quantities and costs before they
are summed to the nation. Each zone of the runs' zone set maps to a state (`inputs/zones/z90/hierarchy.csv`)
and each state to a census division (`inputs/zones/state_groups.csv`). Congestion rent, credits paid
by taxpayers and gas producers stay national, because they have no single division.

The cell covers every run of §3 and §4. The check after it sums each item over the nine divisions and
compares the sum with the national extract of the same run.

In [10]:
CD_RUNS = {}
for d in (STAGE1, STAGE2):
    for b, cs in d.items():
        CD_RUNS.setdefault(b, [])
        CD_RUNS[b] += [c for c in cs if c not in CD_RUNS[b]]
print("runs for the census-division extracts:", sum(len(v) for v in CD_RUNS.values()))
t0 = time.time()
for b, cs in CD_RUNS.items():
    t1 = time.time()
    dst = agr.reduce_cases(b, cs, out=EXTRACTS_AG, verbose=False, fn=agr.reduce_run_cendiv,
                           fname="derived_cendiv.parquet", skip_existing=True)
    print(f"{b}: {len(cs)} runs -> {paths.rel(dst)} ({(time.time() - t1) / 60:.1f} min)", flush=True)
print(f"census-division extracts done in {(time.time() - t0) / 60:.1f} min")

runs for the census-division extracts: 333
smr100: 18 runs -> z-ethan\results\extracts_ag\smr100\derived_cendiv.parquet (0.0 min)


itcfb: 18 runs -> z-ethan\results\extracts_ag\itcfb\derived_cendiv.parquet (0.0 min)


batch2: 8 runs -> z-ethan\results\extracts_ag\batch2\derived_cendiv.parquet (0.0 min)


ratedesign2: 37 runs -> z-ethan\results\extracts_ag\ratedesign2\derived_cendiv.parquet (0.0 min)


batch3: 108 runs -> z-ethan\results\extracts_ag\batch3\derived_cendiv.parquet (0.0 min)


ratedesign: 12 runs -> z-ethan\results\extracts_ag\ratedesign\derived_cendiv.parquet (0.0 min)


step4: 108 runs -> z-ethan\results\extracts_ag\step4\derived_cendiv.parquet (0.0 min)


itcfbm: 24 runs -> z-ethan\results\extracts_ag\itcfbm\derived_cendiv.parquet (0.0 min)


census-division extracts done in 0.0 min


In [11]:
from aglib import ledger as L
worst = {}
for b, cs in CD_RUNS.items():
    cdv = pd.read_parquet(EXTRACTS_AG / b / "derived_cendiv.parquet")
    cdv = cdv[cdv["case"].isin(cs)]
    assert set(cdv["case"]) == set(cs), f"{b}: every run reduced"
    assert cdv["cendiv"].nunique() == 9, f"{b}: nine census divisions"
    nat = cdv.groupby(["case", "item", "t"])["Value"].sum()
    head = nat.index.get_level_values("item")
    only = lambda key: (lambda d: d[d["case"].isin(cs)])(pd.read_parquet(EXTRACTS_AG / b / f"{key}.parquet"))
    der = only("derived_ledger").set_index(["case", "item", "t"])["Value"]
    rev = only("revenue_nat").groupby(["case", "t"])["Value"].sum()
    sc = only("systemcost")
    tb = only("systemcost_techba")
    tb = tb.merge(sc[["case", "sys_costs"]].drop_duplicates(), on=["case", "sys_costs"])
    nc = sc[~sc["sys_costs"].str.contains(L.CREDIT) & (sc["sys_costs"] != L.ACP)
            & sc["sys_costs"].str.match("inv_|op_")]
    pairs = {
        "P_E, P_C, P_R": (nat[head.isin(["P_E", "P_C", "P_R"])], der[der.index.get_level_values("item").isin(["P_E", "P_C", "P_R"])]),
        "revenue": (nat[head.str.startswith("rev|")].groupby(level=["case", "t"]).sum(), rev),
        "techba": (nat[head.str.startswith("tb|")].groupby(level=["case", "t"]).sum(),
                   tb.groupby(["case", "t"])["Value"].sum()),
        "system cost": (nat[head.isin(["sc|inv", "sc|op"])].groupby(level=["case", "t"]).sum(),
                        nc.groupby(["case", "t"])["Value"].sum()),
    }
    for k, (a, n) in pairs.items():
        a, n = a.align(n, join="outer", fill_value=0.0)
        worst[(b, k)] = float(((a - n).abs() / n.abs().groupby(level="case").transform("max").clip(lower=1.0)).max())
W = pd.Series(worst).unstack()
display(W.map(lambda v: f"{v:.1e}"))
assert (W < 1e-6).all().all(), "division sums equal the national extracts"

,"P_E, P_C, P_R",revenue,system cost,techba
batch2,3.3e-08,2.9e-08,5.9e-08,4.4e-08
batch3,5.4e-08,6.6e-08,9.2e-08,7.5e-08
itcfb,3.3e-08,2.9e-08,7.6e-08,5.6e-08
itcfbm,3.3e-08,2.9e-08,6.6e-08,6.8e-08
ratedesign,3.3e-08,2.9e-08,6.6e-08,5.7e-08
ratedesign2,3.3e-08,2.9e-08,9.3e-08,5.6e-08
smr100,3.3e-08,2.9e-08,5.9e-08,7.0e-08
step4,5.4e-08,6.0e-08,9.5e-08,6.9e-08


## 6. The REC trade wedge by origin (stage 1 and the baselines)

The trade wedge, `REC_paid - REC_recv`, is the main term of the RPS gap (`docs/rps_residual_note.md`).
`aglib.reduce.reduce_run_recs` splits it by origin, with the same prices and merge as `reduce_run`:

- **seller_unbound:** the selling state's price for that category is zero, so the buyer's whole price is wedge;
- **double_count:** `eq_REC_Generation` (`reeds/core/setup/c_model.gms`) removes exports from the seller's credit supply, and
  `eq_REC_Requirement` subtracts them again from the RECs that count toward the seller's requirement. The buyer's price is
  then about twice the seller's or more. This part is min(p_dst - p_src, p_src) per REC;
- **trade_limits:** the rest of the price difference on those RECs;
- **voluntary:** RECs sold to the voluntary market, which has no reported price.

The output is `<batch>/derived_recs.parquet`. The check: the four parts sum to `REC_paid - REC_recv` of
`derived_ledger.parquet` in every run and year. The last cell lists the smallest buyer/seller price ratio on
RECs from priced sellers, by run.

In [12]:
t0 = time.time()
for b, cs in STAGE1.items():
    dst = agr.reduce_cases(b, cs, out=EXTRACTS_AG, verbose=False, fn=agr.reduce_run_recs,
                           fname="derived_recs.parquet")      # rebuilt each run (0.6 min), so no file is deleted
    print(f"{b}: {len(cs)} runs -> {paths.rel(dst)}")
print(f"REC extracts done in {(time.time() - t0) / 60:.1f} min")
worst = {}
for b, cs in STAGE1.items():
    rc = pd.read_parquet(EXTRACTS_AG / b / "derived_recs.parquet")
    rc = rc[rc["case"].isin(cs) & rc["item"].str.startswith("wedge|")].groupby(["case", "t"])["Value"].sum()
    der = pd.read_parquet(EXTRACTS_AG / b / "derived_ledger.parquet")
    der = der[der["case"].isin(cs)].pivot_table(index=["case", "t"], columns="item", values="Value")
    ref = der["REC_paid"] - der["REC_recv"]
    a, n = rc.align(ref, join="outer", fill_value=0.0)
    worst[b] = float(((a - n).abs() / n.abs().groupby(level="case").transform("max").clip(lower=1.0)).max())
print({k: f"{v:.1e}" for k, v in worst.items()})
assert max(worst.values()) < 1e-6, "the wedge origins sum to REC_paid - REC_recv"

smr100: 6 runs -> z-ethan\results\extracts_ag\smr100\derived_recs.parquet


itcfb: 2 runs -> z-ethan\results\extracts_ag\itcfb\derived_recs.parquet


batch2: 8 runs -> z-ethan\results\extracts_ag\batch2\derived_recs.parquet


ratedesign2: 1 runs -> z-ethan\results\extracts_ag\ratedesign2\derived_recs.parquet


batch3: 8 runs -> z-ethan\results\extracts_ag\batch3\derived_recs.parquet
REC extracts done in 0.5 min
{'smr100': '1.3e-07', 'itcfb': '9.2e-08', 'batch2': '9.7e-08', 'ratedesign2': '1.1e-07', 'batch3': '1.4e-07'}


In [13]:
mr = pd.concat([pd.read_parquet(EXTRACTS_AG / b / "derived_recs.parquet").query("case in @cs") for b, cs in STAGE1.items()])
mr = mr[(mr["item"] == "min_ratio|seller_priced") & (mr["t"] >= 2026)]
print("smallest buyer/seller price ratio over 2026-2050, RECs from priced sellers:")
display(mr.groupby("case")["Value"].min().round(4).to_frame("min ratio"))

smallest buyer/seller price ratio over 2026-2050, RECs from priced sellers:


,min ratio
case,
ax_eia_p95_c60,2.0000
bx_aj_p75_f50,2.0000
bx_aj_p75_f50_co2cap,2.0000
bx_cop28_p50_f50,2.0000
bx_cop28_p50_f50_co2cap,2.0000
bx_cop28_p75_f50,2.0000
bx_cop28_p75_f50_co2cap,2.0000
bx_eia_p75_f50,2.0000
bx_eia_p75_f50_co2cap,2.0000


## 7. Hourly energy value (stage 1 and the baselines)

`aglib.reduce.reduce_run_hourly` (protocol amendment A17) sums, for each solve year from 2026, season and
hour-ending of the representative timeslices: the load quantity and payment (`load_MWh`, `load_pay`), and for
each owner group its output (`gen_MWh|g`) and the energy value of that output at its zone's load price
(`gen_val|g`). The output is `<batch>/derived_hourly.parquet`. Existing cases are kept (about 15 s per run);
delete the file to rebuild after a change to the reducer. The check: the `load_pay` cells sum to `P_E` of
`derived_ledger.parquet` in every run and year.

In [14]:
t0 = time.time()
for b, cs in STAGE1.items():
    dst = agr.reduce_cases(b, cs, out=EXTRACTS_AG, verbose=False, fn=agr.reduce_run_hourly,
                           fname="derived_hourly.parquet", skip_existing=True)
    print(f"{b}: {len(cs)} runs -> {paths.rel(dst)}")
print(f"hourly extracts done in {(time.time() - t0) / 60:.1f} min")
worst = {}
for b, cs in STAGE1.items():
    hr = pd.read_parquet(EXTRACTS_AG / b / "derived_hourly.parquet")
    lp = hr[hr["case"].isin(cs) & (hr["item"] == "load_pay")].groupby(["case", "t"])["Value"].sum()
    der = pd.read_parquet(EXTRACTS_AG / b / "derived_ledger.parquet")
    pe = der[der["case"].isin(cs) & (der["item"] == "P_E") & (der["t"] >= 2026)].set_index(["case", "t"])["Value"]
    a, n = lp.align(pe, join="outer", fill_value=0.0)
    worst[b] = float(((a - n).abs() / n.abs().clip(lower=1.0)).max())
print({k: f"{v:.1e}" for k, v in worst.items()})
assert max(worst.values()) < 1e-6, "the load_pay cells sum to P_E"

smr100: 6 runs -> z-ethan\results\extracts_ag\smr100\derived_hourly.parquet
itcfb: 2 runs -> z-ethan\results\extracts_ag\itcfb\derived_hourly.parquet
batch2: 8 runs -> z-ethan\results\extracts_ag\batch2\derived_hourly.parquet
ratedesign2: 1 runs -> z-ethan\results\extracts_ag\ratedesign2\derived_hourly.parquet
batch3: 8 runs -> z-ethan\results\extracts_ag\batch3\derived_hourly.parquet
hourly extracts done in 0.0 min
{'smr100': '1.2e-07', 'itcfb': '0.0e+00', 'batch2': '1.1e-07', 'ratedesign2': '6.5e-08', 'batch3': '8.8e-08'}


## 9. The new runs (lab batch `ag`, returned 2026-11-16)

The lab machine ran the 23 cases of `cases_nuclearlearning_ag.csv` (hand-off `agent_comms/2026-11-05_new_runs/`,
final response at commit d09fc70): RUN-3 (7 `recfix_*` runs with `GSw_RECExportFix = 1` and the 2 `recctl_*` controls
with the switch off; the other 5 controls are `ignore = 1`), RUN-1 (6 `run1_*_grow` runs to 2055), stage 2b (6
`nomand_fbR_none_*` baselines) and the 2 P05 screening runs. Each run's `outputs.h5` was copied from
`D:/ag_new_runs/ag_<case>/outputs/` (from `D:/ag_new_runs_diag/` for `run1_smr100_eia_p50_grow`, whose `outputs.h5`
`report_dump.py` wrote on a copy of the run folder, deviation 7 of the response) to `D:/ag_new_runs_outputs/ag/`
in the archive layout, and all 23 copies match the lab's `results/MD5SUMS_outputs.txt`.

This section reads that folder for batch `ag` only: `NEW_ARCHIVE` replaces `NL_REEDS_OUTPUTS` inside `with_archive`
and the old value comes back afterwards. It writes the linear extracts for all 23 runs (the spec rows with
`batches = ag`), and the derived ledger and the REC split by origin (§6) for the 9 RUN-3 runs. The derived
extracts of RUN-1 (solve years to 2055), stage 2b and P05 come later. The checks are those of §3 and §6.

In [15]:
from contextlib import contextmanager
NEW_ARCHIVE = Path("D:/ag_new_runs_outputs")

@contextmanager
def with_archive(root):
    old = os.environ.get(paths.ENV_OUTPUTS)
    os.environ[paths.ENV_OUTPUTS] = str(root)
    try:
        yield
    finally:
        if old is None:
            os.environ.pop(paths.ENV_OUTPUTS, None)
        else:
            os.environ[paths.ENV_OUTPUTS] = old

SPEC_AG = SPEC.copy()
SPEC_AG["batches"] = "ag"
t0 = time.time()
with with_archive(NEW_ARCHIVE):
    AG_ALL = h5io.cases("ag")
    AG_RETURNED = [c for c in AG_ALL if h5io.run_file("ag", c).exists()]
    MAN = h5io.extract(SPEC_AG, batches=["ag"], out=EXTRACTS_AG, verbose=False)
print(f"batch ag: {len(AG_ALL)} casefile columns, {len(AG_RETURNED)} returned; linear extracts in {(time.time() - t0) / 60:.1f} min")
not_run = sorted(set(AG_ALL) - set(AG_RETURNED))
print("not run (ignore = 1):", not_run)
assert len(AG_RETURNED) == 23 and all(c.startswith("recctl_") for c in not_run)
m = MAN[MAN["batch"] == "ag"]
display(m.loc[m["n_cases"] < 23, ["key", "n_cases", "n_missing"]])

batch ag: 28 casefile columns, 23 returned; linear extracts in 0.0 min
not run (ignore = 1): ['recctl_smr100_aj_p50', 'recctl_smr100_cop28_p50', 'recctl_smr100_eia_p50', 'recctl_smr100_iaea_p50', 'recctl_smr100_mck_p50']


,key,n_cases,n_missing
4,co2_price,1,27
17,nuclear_cap_price,13,15


In [16]:
RUN3 = [c for c in AG_RETURNED if c.startswith(("recfix_", "recctl_"))]
assert len(RUN3) == 9
t0 = time.time()
with with_archive(NEW_ARCHIVE):
    agr.reduce_cases("ag", RUN3, out=EXTRACTS_AG, verbose=False, skip_existing=True)
    agr.reduce_cases("ag", RUN3, out=EXTRACTS_AG, verbose=False, fn=agr.reduce_run_recs, fname="derived_recs.parquet")
print(f"RUN-3 derived extracts done in {(time.time() - t0) / 60:.1f} min")
der = pd.read_parquet(EXTRACTS_AG / "ag" / "derived_ledger.parquet")
der = der[der["case"].isin(RUN3)]
chk = der.groupby("case").agg(items=("item", "nunique"), first_t=("t", "min"), last_t=("t", "max"))
display(chk)
assert set(chk.index) == set(RUN3) and (chk["last_t"] == 2050).all()
rc = pd.read_parquet(EXTRACTS_AG / "ag" / "derived_recs.parquet")
rc = rc[rc["case"].isin(RUN3) & rc["item"].str.startswith("wedge|")].groupby(["case", "t"])["Value"].sum()
pv_ = der.pivot_table(index=["case", "t"], columns="item", values="Value")
ref = pv_["REC_paid"] - pv_["REC_recv"]
a, n = rc.align(ref, join="outer", fill_value=0.0)
worst_ag = float(((a - n).abs() / n.abs().groupby(level="case").transform("max").clip(lower=1.0)).max())
print(f"RUN-3: wedge origins vs REC_paid - REC_recv, worst relative difference {worst_ag:.1e}")
assert worst_ag < 1e-6

RUN-3 derived extracts done in 0.2 min


,items,first_t,last_t
case,,,
recctl_fbR_none,13,2010,2050
recctl_smr100_eo_p50,13,2010,2050
recfix_fbR_none,13,2010,2050
recfix_smr100_aj_p50,13,2010,2050
recfix_smr100_cop28_p50,13,2010,2050
recfix_smr100_eia_p50,13,2010,2050
recfix_smr100_eo_p50,13,2010,2050
recfix_smr100_iaea_p50,13,2010,2050
recfix_smr100_mck_p50,13,2010,2050


RUN-3: wedge origins vs REC_paid - REC_recv, worst relative difference 1.2e-07


## 9b. Stage-2b condition baselines and RUN-1 (lab batch `ag`), with the RUN-1 comparison runs

**Stage-2b condition baselines.** `nomand_fbR_none_{gaslo,gashi,demhi,relo,rehi,translim}` are `fbR_none` under
each of the six market conditions. Ethan chose on 2026-11-04 to add them (review page item 9,
`stage2b-baselines`, "newruns"), so stage 2b can measure the programme alone. They get the same derived extracts
as the other baselines: the ledger items, the REC wedge by origin, and the census-division items (notebook 26).

**RUN-1.** `run1_smr100_{t}_p50_grow` keeps the six P50 mandates growing to 2055 at their 2041–2050 pace (item 6).
Its comparison runs are paper 1's `rate_design_hz_{t}_p50_ext` (batch `ratedesign`, public archive, case names
`hz_{t}_p50_ext`): the same samples to 2055 with the fleet flat after 2050. Both get the ledger items and the
REC wedge. ReEDS solves one year at a time with no foresight, so through 2050 each RUN-1 run must equal its `_ext`
run. The cell checks this on every derived item. No readout after 2050 is computed here: what RUN-1 reports waits
for review page item 21.

**Canada exports in stress slices (amendment A11, O28).** `can_exports_stress` is missing in all 23 new runs. The
code shows why: `reeds/core/solve/2_temporal_params.gms` line 296 reads `can_exports_h_frac` only from the
representative-period folder (`%temporal_inputs%`), with no `stress%stress_year%` include, so `can_exports_h` is
zero in every stress slice and the reported parameter is empty. The capacity part of export payments is zero by
construction (O37).

In [17]:
NOMAND = [c for c in AG_RETURNED if c.startswith("nomand_fbR_none_")]
RUN1 = [c for c in AG_RETURNED if c.startswith("run1_")]
EXT1 = {c: c.replace("run1_smr100_", "hz_").replace("_grow", "_ext") for c in RUN1}
assert len(NOMAND) == 6 and len(RUN1) == 6
t0 = time.time()
with with_archive(NEW_ARCHIVE):
    for fn, fname in ((None, "derived_ledger.parquet"), (agr.reduce_run_recs, "derived_recs.parquet")):
        agr.reduce_cases("ag", NOMAND + RUN1, out=EXTRACTS_AG, verbose=False, fn=fn, fname=fname, skip_existing=True)
    agr.reduce_cases("ag", NOMAND, out=EXTRACTS_AG, verbose=False, fn=agr.reduce_run_cendiv,
                     fname="derived_cendiv.parquet", skip_existing=True)
for fn, fname in ((None, "derived_ledger.parquet"), (agr.reduce_run_recs, "derived_recs.parquet")):
    agr.reduce_cases("ratedesign", list(EXT1.values()), out=EXTRACTS_AG, verbose=False, fn=fn, fname=fname,
                     skip_existing=True)
print(f"stage-2b baselines, RUN-1 and _ext derived extracts done in {(time.time() - t0) / 60:.1f} min")

der_ag = pd.read_parquet(EXTRACTS_AG / "ag" / "derived_ledger.parquet")
der_rd = pd.read_parquet(EXTRACTS_AG / "ratedesign" / "derived_ledger.parquet")
chk = pd.concat([der_ag[der_ag["case"].isin(NOMAND + RUN1)], der_rd[der_rd["case"].isin(EXT1.values())]])
chk = chk.groupby("case").agg(items=("item", "nunique"), first_t=("t", "min"), last_t=("t", "max"))
display(chk)
assert (chk.loc[NOMAND, "last_t"] == 2050).all() and (chk.loc[RUN1 + list(EXT1.values()), "last_t"] == 2055).all()

cdv = pd.read_parquet(EXTRACTS_AG / "ag" / "derived_cendiv.parquet", columns=["case"])
assert set(NOMAND) <= set(cdv["case"])
rc = pd.read_parquet(EXTRACTS_AG / "ag" / "derived_recs.parquet")
rc = rc[rc["case"].isin(NOMAND + RUN1) & rc["item"].str.startswith("wedge|")].groupby(["case", "t"])["Value"].sum()
pv_ = der_ag[der_ag["case"].isin(NOMAND + RUN1)].pivot_table(index=["case", "t"], columns="item", values="Value")
ref = pv_["REC_paid"] - pv_["REC_recv"]
a, n = rc.align(ref, join="outer", fill_value=0.0)
worst_nr = float(((a - n).abs() / n.abs().groupby(level="case").transform("max").clip(lower=1.0)).max())
print(f"stage-2b baselines and RUN-1: wedge origins vs REC_paid - REC_recv, worst relative difference {worst_nr:.1e}")
assert worst_nr < 1e-6

# RUN-1 against its _ext run, every derived item, solve years to 2050
rows = []
for r1, ex in EXT1.items():
    x = der_ag[der_ag["case"] == r1].set_index(["item", "t"])["Value"]
    y = der_rd[der_rd["case"] == ex].set_index(["item", "t"])["Value"]
    x, y = x.align(y, join="outer", fill_value=0.0)
    for lab, sel in (("2026-2050", lambda s: s[(s.index.get_level_values("t") >= 2026) & (s.index.get_level_values("t") <= 2050)]),
                     ("2051-2055", lambda s: s[s.index.get_level_values("t") > 2050])):
        dx, scale = (sel(x) - sel(y)).abs(), sel(y).abs().groupby(level="item").transform("max").clip(lower=1.0)
        rel = dx / scale
        rows.append(dict(run1=r1, ext=ex, years=lab, items=int(rel.index.get_level_values("item").nunique()),
                         worst_rel=float(rel.max()), worst_item=rel.idxmax()[0] if len(rel) else None,
                         items_over_1e_4=int((rel.groupby(level="item").max() > 1e-4).sum())))
R1CHK = pd.DataFrame(rows)
R1CHK.to_csv(paths.exports("20") / "run1_vs_ext_check.csv", index=False)
print(R1CHK.to_string(index=False))


stage-2b baselines, RUN-1 and _ext derived extracts done in 0.0 min


,items,first_t,last_t
case,,,
hz_aj_p50_ext,13,2010,2055
hz_cop28_p50_ext,13,2010,2055
hz_eia_p50_ext,13,2010,2055
hz_eo_p50_ext,13,2010,2055
hz_iaea_p50_ext,13,2010,2055
hz_mck_p50_ext,13,2010,2055
nomand_fbR_none_demhi,13,2010,2050
nomand_fbR_none_gashi,13,2010,2050
nomand_fbR_none_gaslo,13,2010,2050


stage-2b baselines and RUN-1: wedge origins vs REC_paid - REC_recv, worst relative difference 1.9e-07


                      run1              ext     years  items  worst_rel worst_item  items_over_1e_4
  run1_smr100_eia_p50_grow   hz_eia_p50_ext 2026-2050     13   0.000615   CR_C_vsc                6
  run1_smr100_eia_p50_grow   hz_eia_p50_ext 2051-2055     13   1.000000     P_H2_E               12
   run1_smr100_aj_p50_grow    hz_aj_p50_ext 2026-2050     13   0.000402   CR_C_vsc                5
   run1_smr100_aj_p50_grow    hz_aj_p50_ext 2051-2055     13   0.106690   REC_recv               12
 run1_smr100_iaea_p50_grow  hz_iaea_p50_ext 2026-2050     13   0.002156   CR_C_vsc                6
 run1_smr100_iaea_p50_grow  hz_iaea_p50_ext 2051-2055     13   0.249425        P_R               12
  run1_smr100_mck_p50_grow   hz_mck_p50_ext 2026-2050     13   0.003995   CR_C_vsc                8
  run1_smr100_mck_p50_grow   hz_mck_p50_ext 2051-2055     13   0.213773   REC_recv               12
run1_smr100_cop28_p50_grow hz_cop28_p50_ext 2026-2050     13   0.001847   CR_C_vsc                2


## 9c. The REC trade wedge by seller group, for every ledger run (amendment A23)

Ethan decided on 2026-11-03 (review page items 2 and 4) that the selling generators receive the parts of the REC
trade wedge from sellers whose own rule does not bind and from trade limits; Prof. Davies affirmed it on
2026-11-09. Amendment A23 puts that in the ledger: each part goes to the owner group of the selling technology.
`aglib.reduce.reduce_run_recs` now also writes `wedge_g|<origin>|<group>`, so this cell reduces the REC split
again for every run that has derived ledger items (all batches; stage 1, the baselines, stage 2, RUN-1, RUN-3 and
the `_ext` runs). The cell checks that the group items sum to the national origin items, and that the origins
sum to `REC_paid − REC_recv`.

In [18]:
t0 = time.time()
n_cases = {}
for bdir in sorted(EXTRACTS_AG.iterdir()):
    f = bdir / "derived_ledger.parquet"
    if not f.exists():
        continue
    cs = sorted(pd.read_parquet(f, columns=["case"])["case"].unique())
    if bdir.name == "ag":
        with with_archive(NEW_ARCHIVE):
            agr.reduce_cases("ag", cs, out=EXTRACTS_AG, verbose=False, fn=agr.reduce_run_recs, fname="derived_recs.parquet")
    else:
        agr.reduce_cases(bdir.name, cs, out=EXTRACTS_AG, verbose=False, fn=agr.reduce_run_recs, fname="derived_recs.parquet")
    n_cases[bdir.name] = len(cs)
print(f"REC split by seller group: {sum(n_cases.values())} runs in {(time.time() - t0) / 60:.1f} min", n_cases)

worst_g, worst_o = 0.0, 0.0
for b in n_cases:
    rc = pd.read_parquet(EXTRACTS_AG / b / "derived_recs.parquet")
    der = pd.read_parquet(EXTRACTS_AG / b / "derived_ledger.parquet")
    assert set(der["case"]) <= set(rc["case"]), (b, set(der["case"]) - set(rc["case"]))
    nat = rc[rc["item"].str.startswith("wedge|")].assign(o=lambda d: d["item"].str.split("|").str[1])
    grp = rc[rc["item"].str.startswith("wedge_g|")].assign(o=lambda d: d["item"].str.split("|").str[1])
    a, n = (grp.groupby(["case", "t", "o"])["Value"].sum().align(nat.groupby(["case", "t", "o"])["Value"].sum(),
                                                                  join="outer", fill_value=0.0))
    scale = n.abs().groupby(level="case").transform("max").clip(lower=1.0)
    worst_g = max(worst_g, float(((a - n).abs() / scale).max()))
    pv_ = der.pivot_table(index=["case", "t"], columns="item", values="Value")
    ref = (pv_["REC_paid"] - pv_["REC_recv"])
    w, r = nat.groupby(["case", "t"])["Value"].sum().align(ref, join="outer", fill_value=0.0)
    worst_o = max(worst_o, float(((w - r).abs() / r.abs().groupby(level="case").transform("max").clip(lower=1.0)).max()))
print(f"group items vs origin items, worst relative difference {worst_g:.1e}; origins vs REC_paid - REC_recv {worst_o:.1e}")
assert worst_g < 1e-9 and worst_o < 1e-6


REC split by seller group: 360 runs in 6.5 min {'ag': 21, 'batch2': 8, 'batch3': 108, 'itcfb': 18, 'itcfbm': 24, 'ratedesign': 18, 'ratedesign2': 37, 'smr100': 18, 'step4': 108}


group items vs origin items, worst relative difference 5.0e-17; origins vs REC_paid - REC_recv 2.0e-07


## 10. What reads these extracts

Notebook 21 (`21_ag_ledger`) builds the stage-1 ledger and the closure report from both kinds of
extract. Notebook 22 (`22_ag_baselines`) verifies the 19 baselines. Notebook 24 builds the stage-2
ledger. Notebook 25 (`25_ag_review`) reads the REC extracts of §6. Notebook 23 (`23_ag_markets`) reads the hourly extracts of §7. Notebook 26 (`26_ag_cendiv`) reads
the census-division extracts. None of them opens an h5 file.

Notebook 28 (`28_ag_run3`) reads the batch-`ag` extracts of §9 for RUN-3.